# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/girishpatil935/ML_Internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# Structured Content Performance Archetypes: A Data-Driven Action Playbook

## Abstract

This research asks what measurable performance archetypes exist across a large content inventory and how those patterns can support content-review decisions. Using the FlyRank internship warehouse, the analysis aggregates March 2026 Google Search Console measurements at the client-content level and represents each page using impressions, clicks, CTR, average position, and position volatility. A transparent CTR opportunity baseline is compared with an unsupervised K-Means clustering approach, with client-grouped validation and explicit leakage checks used to assess whether the observed structure is stable and whether the evaluation is honest. The analysis produced three descriptive performance clusters and a ranked review queue of 1,368 pages with positive baseline opportunity scores, with high-visibility, lower-CTR pages receiving the highest review priority. The resulting playbook is intended as decision support for human review rather than as an automated system for rewriting, pruning, merging, or predicting future search performance.

### Research question

**What performance archetypes exist across the content inventory, and how can those observed patterns support content-review prioritization?**

The analysis focuses on measurable search-performance patterns rather than semantic content categories because the available warehouse does not contain article text.

### Decision supported

The analysis is designed to help a content or SEO team decide:

- Which pages deserve review first?
- What observed performance pattern should a reviewer investigate?
- Which pages show measurable visibility but a potential CTR opportunity?
- Which pages require deeper investigation before any content action is considered?

The output is therefore a **review-prioritization framework**, not an automated content decision system.

### Scope

The research uses March 2026 as the fixed development window for the page-level feature vector. The analysis combines:

1. A transparent CTR-based baseline for ranking review opportunities.
2. K-Means clustering to provide descriptive performance archetype context.
3. Client-grouped validation to test the stability of the clustering structure across unseen clients.
4. A human-reviewed action playbook that maps observed patterns to investigation priorities.

The central question is descriptive and operational: **what patterns are present in the observed data, and how can they be used responsibly to organize human review?**

In [1]:
research_question = {
    "question": (
        "What performance archetypes exist across the content inventory, "
        "and how can those observed patterns support content-review prioritization?"
    ),
    "development_window": "March 2026",
    "unit_of_analysis": "client-content pair",
    "baseline": "CTR opportunity by search-position tier",
    "model": "K-Means clustering",
    "decision_output": "human-reviewed content action prioritization"
}

for key, value in research_question.items():
    print(f"{key}: {value}")

question: What performance archetypes exist across the content inventory, and how can those observed patterns support content-review prioritization?
development_window: March 2026
unit_of_analysis: client-content pair
baseline: CTR opportunity by search-position tier
model: K-Means clustering
decision_output: human-reviewed content action prioritization


### Dataset and scope

This research uses the **FlyRank ML Internship warehouse**, accessed through the `FlyRank/internship-warehouse` dataset. The warehouse contains daily content-performance observations covering approximately 79 million rows, with pseudonymized client and content identifiers.

The primary analytical table is the daily content-performance fact table:

`fact_content_daily_performance`

Its analytical grain is:

**report_date × client_hash_id × content_hash_id**

The available measurements include Google Search Console performance, Google Analytics measurements, traffic-source sessions, AI-referral measurements, and engagement signals.

### Time windows

The warehouse covers daily observations from **January 27, 2025 through June 30, 2026**.

Different windows were used for different purposes:

- **March 2026** — fixed development window used to construct the client-content feature vector and develop the clustering analysis.
- **April 2026** — used only for the Week-5 leakage check and excluded from the final feature vector.
- **June 2026** — treated as a later final month and kept sealed from model development.
- Earlier historical observations were used for contextual understanding where appropriate, but the final clustering features were constructed from March 2026 only.

The March window was chosen as a fixed development period so that feature construction, baseline scoring, clustering, and validation could be evaluated consistently without using later observations.

### Data availability

Google Search Console coverage is uneven across the warehouse. Some client-content observations have GSC measurements while others do not.

For the clustering analysis, only observations where `gsc_data_available` was true were used to construct the search-performance feature vector.

This means the analysis describes the portion of the content inventory with available GSC measurements rather than the entire warehouse population.

### Features used

The final clustering feature vector contains five March 2026 measurements:

- **Impressions** — total observed Google Search impressions.
- **Clicks** — total observed Google Search clicks.
- **CTR** — clicks divided by impressions.
- **Average position** — average observed search position, excluding zero-valued no-data observations.
- **Position volatility** — sample standard deviation of observed daily search position.

Impressions and clicks were log-transformed before clustering because their distributions were highly right-skewed. Missing feature values were median-imputed and the resulting features were standardized.

### What was excluded

The following information was excluded from the model features:

- `client_hash_id` and `content_hash_id` as numeric predictors, because they are identifiers rather than performance measurements.
- Report dates and month fields, because the development window was fixed.
- Future observations such as April 2026 measurements, to prevent temporal leakage.
- Trend-derived labels or fields, because they are derived from performance outcomes.
- Baseline action scores, because they are downstream decision outputs rather than independent features.
- GSC availability flags, because availability describes data coverage rather than page performance.
- Google Analytics, traffic-source, AI-referral, and engagement fields, because the selected clustering task focuses specifically on observed search-performance patterns.

### Public-safety framing

The analysis uses pseudonymized identifiers only for grouping, joining, validation, and traceability. Client names, URLs, private search queries, and other identifying information are not included in the research paper or public artifacts.

The reported findings are aggregate or pseudonymized and are intended to describe measured performance patterns rather than individual clients or private properties.

In [3]:
import os
import getpass
import duckdb
import pandas as pd

def get_hf_token():
    token = os.environ.get("HF_TOKEN")

    if token:
        return token

    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")

        if token:
            return token
    except Exception:
        pass

    return getpass.getpass(
        "Paste your Hugging Face READ token (hf_...): "
    )

HF_TOKEN = get_hf_token()

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf
(
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT_DAILY = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/**/*.parquet'"
    f")"
)

print("DuckDB connection ready.")
print("Warehouse relation ready.")

DuckDB connection ready.
Warehouse relation ready.


In [5]:
# Verify the warehouse scope used in the paper

import duckdb
import pandas as pd

# Reuse the existing DuckDB connection if available.
# If the capstone notebook has not created one yet, run the
# warehouse setup cell from the earlier notebooks first.

try:
    con
except NameError:
    raise RuntimeError(
        "DuckDB connection `con` is not defined. "
        "Run the warehouse setup cell before this section."
    )

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT_DAILY = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/**/*.parquet'"
    f")"
)

data_summary = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items
    FROM {FACT_DAILY}
""").df()

display(data_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,first_date,last_date,clients,content_items
0,78835655,2025-01-27,2026-06-30,70,427292


### Method overview

The analysis follows a two-part approach: a transparent rule-based baseline provides a review-priority score, while K-Means clustering provides descriptive performance archetypes.

The workflow is:

1. Aggregate March 2026 GSC observations to the client-content level.
2. Construct five search-performance features.
3. Apply transformations, imputation, and standardization.
4. Establish a transparent CTR-opportunity baseline.
5. Fit K-Means models across candidate cluster counts.
6. Select the final cluster count using silhouette, cluster size, stability, and interpretability.
7. Validate the clustering structure using a client-grouped holdout.
8. Audit for temporal, identifier, label, and preprocessing leakage.
9. Combine the baseline priority score with cluster context in the final action playbook.

### Feature construction

Each client-content pair is represented using five March 2026 search-performance measurements:

| Feature | Definition |
|---|---|
| Impressions | Sum of observed GSC impressions during March |
| Clicks | Sum of observed GSC clicks during March |
| CTR | Total clicks divided by total impressions |
| Average position | Mean non-zero daily GSC average position |
| Position volatility | Sample standard deviation of non-zero daily average position |

Impressions and clicks are log-transformed before clustering because their distributions are strongly right-skewed. Missing feature values are median-imputed and the resulting features are standardized.

Identifiers are retained only for grouping, traceability, and validation and are not used as numeric predictors.

### Baseline

The baseline identifies pages with meaningful search visibility and a potential CTR opportunity.

For each average-position tier, the expected CTR is defined as the median observed CTR within that tier:

- Positions 1–3
- Positions 4–10
- Positions 11–20
- Positions 20+

For pages with at least 500 impressions, the baseline priority score is:

`log(1 + impressions) × max(expected CTR − observed CTR, 0)`

Pages below the 500-impression threshold receive a score of zero.

This baseline is intentionally transparent: it provides a simple review-priority benchmark rather than a prediction of future performance.

### Clustering model

Because there is no ground-truth content-archetype label in the warehouse, the main model is unsupervised **K-Means clustering**.

Candidate values of K from 2 through 8 were evaluated using:

- Silhouette score
- Smallest cluster size
- Cluster stability
- Interpretability of cluster profiles

The final analysis uses **K = 3**. The cluster labels are descriptive rather than ground-truth classes.

The resulting archetypes are interpreted from their observed feature profiles only:

- **Low-visibility, unstable pages**
- **High-visibility performance pages**
- **Low-visibility, relatively well-ranked pages**

### Validation design

The primary validation uses a **client-grouped 80/20 split**.

Clients, rather than individual rows, determine the split so that a client cannot appear in both development and holdout data. This provides a more honest test of whether the observed clustering structure transfers to previously unseen clients.

The development set is used to fit preprocessing and K-Means. The same fitted transformations and cluster model are then applied to the holdout set.

A random row split was also examined as a comparison, but it allowed substantial client overlap between development and holdout and was therefore not treated as the primary generalization test.

### Leakage checks

The analysis explicitly checked for:

- Future-window leakage from observations after March 2026.
- Identifier leakage from client and content IDs.
- Leakage from trend-derived labels or downstream decision fields.
- Preprocessing leakage between development and holdout data.
- Leakage from the baseline action score into the clustering features.

April 2026 measurements were used only as a separate leakage check and were excluded from the final clustering feature vector.

The final clustering features contain only March 2026 observed measurements.

### Label definition

The clustering task has **no supervised label**. K-Means creates descriptive groups from the observed feature space.

The baseline score is also not a target label. It is an independent review-prioritization rule based on observed impressions, CTR, and position tier.

Therefore, neither the clusters nor the baseline score should be interpreted as ground-truth classifications or predictions of future search performance.

In [6]:
# Build the March 2026 client-content feature vector

feature_vector = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions,

        SUM(gsc_clicks) AS clicks,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN
                100.0 * SUM(gsc_clicks)
                / SUM(gsc_impressions)
            ELSE NULL
        END AS ctr,

        AVG(
            NULLIF(gsc_avg_position, 0)
        ) AS avg_position,

        STDDEV_SAMP(
            NULLIF(gsc_avg_position, 0)
        ) AS position_volatility

    FROM {FACT_DAILY}

    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
      AND gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print("March client-content feature vectors:", len(feature_vector))
print(
    "Unique clients:",
    feature_vector["client_hash_id"].nunique()
)

display(
    feature_vector[
        [
            "impressions",
            "clicks",
            "ctr",
            "avg_position",
            "position_volatility"
        ]
    ].describe()
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March client-content feature vectors: 176738
Unique clients: 47


,impressions,clicks,ctr,avg_position,position_volatility
count,176738.000000,176738.000000,176738.000000,175304.000000,161557.000000
mean,1587.986675,4.650002,0.459397,17.050555,9.292292
std,5431.337724,26.722649,3.775992,18.333942,9.955904
min,1.000000,0.000000,0.000000,0.101639,0.000000
25%,20.000000,0.000000,0.000000,5.500000,2.407726
50%,173.000000,0.000000,0.000000,9.000000,5.382745
75%,1039.000000,2.000000,0.215796,22.000000,13.273467
max,617124.000000,5668.000000,100.000000,309.000000,238.294985


In [7]:
cluster_features = [
    "impressions",
    "clicks",
    "ctr",
    "avg_position",
    "position_volatility"
]

excluded_from_features = [
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "month",
    "future April 2026 measurements",
    "trend-derived labels",
    "baseline_action_score",
    "GSC availability flags",
    "GA4 / traffic-source / AI-referral fields"
]

print("Final clustering features:")
for feature in cluster_features:
    print(" -", feature)

print("\nExcluded feature categories:")
for item in excluded_from_features:
    print(" -", item)

print(
    "\nIdentifier leakage detected:",
    any(
        identifier in cluster_features
        for identifier in ["client_hash_id", "content_hash_id"]
    )
)

Final clustering features:
 - impressions
 - clicks
 - ctr
 - avg_position
 - position_volatility

Excluded feature categories:
 - client_hash_id
 - content_hash_id
 - report_date
 - month
 - future April 2026 measurements
 - trend-derived labels
 - baseline_action_score
 - GSC availability flags
 - GA4 / traffic-source / AI-referral fields

Identifier leakage detected: False


### Results overview

The clustering model and the baseline serve different purposes, so they are not compared using a single accuracy score.

The **baseline** is a transparent review-prioritization rule. It identifies pages with meaningful search visibility and observed CTR below the typical CTR for their search-position tier.

The **K-Means model** is an unsupervised descriptive model. It groups pages according to their multidimensional search-performance profiles.

The comparison therefore focuses on:

- Baseline opportunity coverage and ranking behaviour.
- Cluster separation on development and holdout data.
- Stability of the three-cluster structure under a client-grouped holdout.
- Whether the model adds useful performance context beyond the transparent baseline.

### Validated clustering result

The three-cluster solution produced:

- Development silhouette: approximately **0.40**
- Client-grouped holdout silhouette: approximately **0.39**
- Holdout Adjusted Rand Index (ARI): approximately **0.73**

The similar development and holdout silhouette values indicate that the three-cluster structure remained reasonably consistent when evaluated on previously unseen clients. The ARI provides an additional measure of agreement between development-derived and holdout cluster assignments.

These metrics describe the stability of the clustering structure. They do **not** demonstrate that the clusters predict future traffic, rankings, or business outcomes.

### Cluster profiles

The three observed archetypes were:

| Archetype | Observed pattern |
|---|---|
| Low-visibility, unstable pages | Low visibility, weaker observed ranking, and higher position volatility |
| High-visibility performance pages | Higher visibility, stronger median CTR, stronger ranking, and lower volatility |
| Low-visibility, relatively well-ranked pages | Low visibility despite relatively strong median average position |

The clusters are descriptive groupings rather than ground-truth content categories.

### Baseline relationship

The baseline provides a simpler and more transparent prioritization mechanism, while clustering adds multidimensional context.

The final action playbook therefore uses the baseline score as the primary ranking signal and the cluster assignment as contextual information for human review.

This means the analysis does not claim that K-Means outperforms the baseline. Instead, the two methods answer different questions:

**Baseline:** Which pages have a measurable CTR review opportunity?

**Clustering:** What broader search-performance pattern does each candidate belong to?

### Key result

The analysis produced a practical decision-support workflow in which measurable search visibility is used to prioritize review, while descriptive archetypes provide additional context. The resulting recommendations remain subject to human review and additional evidence such as search intent, SERP context, technical status, business value, freshness, and cannibalization.

In [8]:
# Reconstruct the transparent baseline from the March feature vector

import numpy as np
import pandas as pd

results_df = feature_vector.copy()

def position_bucket(position):
    if pd.isna(position):
        return "unknown"
    elif position <= 3:
        return "1-3"
    elif position <= 10:
        return "4-10"
    elif position <= 20:
        return "11-20"
    else:
        return "20+"

results_df["position_bucket"] = (
    results_df["avg_position"].apply(position_bucket)
)

expected_ctr = (
    results_df[
        results_df["position_bucket"] != "unknown"
    ]
    .groupby("position_bucket")["ctr"]
    .median()
)

results_df["expected_ctr"] = (
    results_df["position_bucket"].map(expected_ctr)
)

results_df["ctr_gap"] = (
    results_df["expected_ctr"] - results_df["ctr"]
).clip(lower=0)

results_df["baseline_score"] = np.where(
    results_df["impressions"] >= 500,
    np.log1p(results_df["impressions"])
    * results_df["ctr_gap"],
    0
)

baseline_opportunity_count = int(
    (results_df["baseline_score"] > 0).sum()
)

baseline_opportunity_rate = float(
    (results_df["baseline_score"] > 0).mean()
)

print("Baseline opportunity pages:", baseline_opportunity_count)
print(
    "Baseline opportunity rate:",
    round(baseline_opportunity_rate * 100, 3),
    "%"
)

print(
    "Highest baseline score:",
    round(results_df["baseline_score"].max(), 6)
)

Baseline opportunity pages: 1368
Baseline opportunity rate: 0.774 %
Highest baseline score: 0.987203


## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
